In [3]:
import os
import rasterio
import numpy as np
import geopandas as gpd
import pandas as pd


# -------- USER INPUTS --------
THRESH = 0.3  # meters; depth >= THRESH => flood-affected

rasters = {
    "5-yr":   r"D:\Phd Research\Final_Raster\5yr_compound_flood_stat.tif",
    "20-yr":  r"D:\Phd Research\Final_Raster\20yr_compound_flood_stat.tif",
    "50-yr":  r"D:\Phd Research\Final_Raster\50-yr_compound_flood_stat.tif",
    "100-yr": r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif",
}

ci_layers = {
    "Fire Station":      r"D:\Phd Research\GIS\Shape\Critical Infrastructure\Fire_stn_study_area.zip",
    "Hospital":          r"D:\Phd Research\GIS\Shape\Critical Infrastructure\hospital_study_area.zip",
    "National Shelter":  r"D:\Phd Research\GIS\Shape\Critical Infrastructure\National_shelter_study_area.zip",
    "School":            r"D:\Phd Research\GIS\Shape\Critical Infrastructure\School_study_area.shp.zip",
}

# Optional: where to save the summary table
out_csv = r"D:\Phd Research\Final_Raster\CI_flood_safe_counts_0.3mthreshold.csv"

# -------------------------------------------

def read_ci(path):
    """Read CI points from (zipped) shapefile."""
    p = path
    if p.lower().endswith(".zip") and not p.lower().startswith("zip://"):
        p = f"zip://{p}"
    gdf = gpd.read_file(p)
    # Ensure we have point geometries
    if gdf.empty:
        raise RuntimeError(f"No features found in {path}")
    if not gdf.geometry.geom_type.isin(["Point", "MultiPoint"]).any():
        # Try to explode if multipoints exist
        gdf = gdf.explode(index_parts=False)
        if not gdf.geometry.geom_type.isin(["Point"]).any():
            raise RuntimeError(f"Non-point geometries in {path}. Please provide point CI data.")
    return gdf

def sample_raster_values_at_points(raster_path, gdf_points):
    """Return np.array of sampled raster values at point locations."""
    with rasterio.open(raster_path) as src:
        r_crs = src.crs
        nodata = src.nodata
        # Reproject CI to raster CRS
        pts = gdf_points.to_crs(r_crs).geometry
        coords = [(pt.x, pt.y) for pt in pts]
        # Use the dataset’s sample() method instead of rasterio.sample
        vals = [v[0] for v in src.sample(coords)]
    return np.array(vals, dtype="float64"), nodata


def classify_flooded(values, nodata, thresh):
    """Return a boolean array: True = flood-affected, False = safe."""
    # Treat NoData or non-finite as safe
    is_valid = np.isfinite(values)
    if nodata is not None and np.isfinite(nodata):
        is_valid &= (values != nodata)
    flooded = np.zeros(values.shape, dtype=bool)
    flooded[is_valid] = values[is_valid] >= thresh
    return flooded

# ---- Main analysis ----
records = []

for ci_name, ci_path in ci_layers.items():
    ci_gdf = read_ci(ci_path)
    n_points = len(ci_gdf)

    for scen, rpath in rasters.items():
        vals, nodata = sample_raster_values_at_points(rpath, ci_gdf)
        flooded = classify_flooded(vals, nodata, THRESH)
        n_flood = int(flooded.sum())
        n_safe = int(n_points - n_flood)

        records.append({
            "CI Type": ci_name,
            "Scenario": scen,
            "Total CI": n_points,
            "Flood-affected (>= {:.2f} m)".format(THRESH): n_flood,
            "Flood-safe (< {:.2f} m)".format(THRESH): n_safe
        })

# Make summary table
df = pd.DataFrame.from_records(records)
# Optional: sort
df = df.sort_values(by=["CI Type", "Scenario"])

# Save CSV
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df.to_csv(out_csv, index=False)

# Pretty print
pd.set_option("display.max_rows", 200)
print(df)
print("\nSaved:", out_csv)


             CI Type Scenario  Total CI  Flood-affected (>= 0.30 m)  \
3       Fire Station   100-yr        62                          62   
1       Fire Station    20-yr        62                          52   
0       Fire Station     5-yr        62                          32   
2       Fire Station    50-yr        62                          59   
7           Hospital   100-yr        22                          22   
5           Hospital    20-yr        22                          14   
4           Hospital     5-yr        22                           9   
6           Hospital    50-yr        22                          19   
11  National Shelter   100-yr        47                          47   
9   National Shelter    20-yr        47                          36   
8   National Shelter     5-yr        47                          15   
10  National Shelter    50-yr        47                          43   
15            School   100-yr       171                         171   
13    

In [4]:
import os
import rasterio
import numpy as np
import geopandas as gpd
import pandas as pd

# -------- USER INPUTS --------
THRESH = 0.1  # meters; depth >= THRESH => flood-affected

# Set True if the NOAA raster is in feet (will convert to meters)
NOAA_IN_FEET = False  # change to True if needed

rasters = {
    "SLR (Our)":  r"D:\Phd Research\Final_Raster\Inundation_only_for_SLR.tif",
    "SLR (NOAA)": r"C:\Users\sahad2\Research_data\SLR_Depth_Raster_NOAA\SLR_depth_3_5ft_tx_200m_UTM_m.tif",
}

ci_layers = {
    "Fire Station":      r"D:\Phd Research\GIS\Shape\Critical Infrastructure\Fire_stn_study_area.zip",
    "Hospital":          r"D:\Phd Research\GIS\Shape\Critical Infrastructure\hospital_study_area.zip",
    "National Shelter":  r"D:\Phd Research\GIS\Shape\Critical Infrastructure\National_shelter_study_area.zip",
    "School":            r"D:\Phd Research\GIS\Shape\Critical Infrastructure\School_study_area.shp.zip",
}

# Output
out_csv = r"D:\Phd Research\Final_Raster\CI_flood_safe_counts_SLR_vs_NOAA_0p3m.csv"

# -------------------------------------------

def read_ci(path):
    """Read CI points from (zipped) shapefile."""
    p = path
    if p.lower().endswith(".zip") and not p.lower().startswith("zip://"):
        p = f"zip://{p}"
    gdf = gpd.read_file(p)
    # Ensure we have point geometries
    if gdf.empty:
        raise RuntimeError(f"No features found in {path}")
    if not gdf.geometry.geom_type.isin(["Point", "MultiPoint"]).any():
        gdf = gdf.explode(index_parts=False)
        if not gdf.geometry.geom_type.isin(["Point"]).any():
            raise RuntimeError(f"Non-point geometries in {path}. Please provide point CI data.")
    return gdf

def sample_raster_values_at_points(raster_path, gdf_points):
    """Return np.array of sampled raster values at point locations."""
    with rasterio.open(raster_path) as src:
        r_crs = src.crs
        nodata = src.nodata
        # Reproject CI to raster CRS
        pts = gdf_points.to_crs(r_crs).geometry
        coords = [(pt.x, pt.y) for pt in pts]
        # sample band-1 values
        vals = [v[0] for v in src.sample(coords)]
    return np.array(vals, dtype="float64"), nodata

def classify_flooded(values, nodata, thresh):
    """Return a boolean array: True = flood-affected, False = safe."""
    # Treat NoData or non-finite as safe
    is_valid = np.isfinite(values)
    if nodata is not None and np.isfinite(nodata):
        is_valid &= (values != nodata)
    flooded = np.zeros(values.shape, dtype=bool)
    flooded[is_valid] = values[is_valid] >= thresh
    return flooded

# ---- Main analysis ----
records = []

for ci_name, ci_path in ci_layers.items():
    ci_gdf = read_ci(ci_path)
    n_points = len(ci_gdf)

    for scen, rpath in rasters.items():
        vals, nodata = sample_raster_values_at_points(rpath, ci_gdf)

        # Optional unit conversion for NOAA (feet -> meters)
        if scen == "SLR (NOAA)" and NOAA_IN_FEET:
            vals = vals * 0.3048

        flooded = classify_flooded(vals, nodata, THRESH)
        n_flood = int(flooded.sum())
        n_safe = int(n_points - n_flood)

        records.append({
            "CI Type": ci_name,
            "Scenario": scen,
            "Total CI": n_points,
            f"Flood-affected (>= {THRESH:.2f} m)": n_flood,
            f"Flood-safe (< {THRESH:.2f} m)": n_safe
        })

# Make summary table
df = pd.DataFrame.from_records(records)
df = df.sort_values(by=["CI Type", "Scenario"])

# Save CSV
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df.to_csv(out_csv, index=False)

# Pretty print
pd.set_option("display.max_rows", 200)
print(df)
print("\nSaved:", out_csv)


            CI Type    Scenario  Total CI  Flood-affected (>= 0.10 m)  \
1      Fire Station  SLR (NOAA)        62                           0   
0      Fire Station   SLR (Our)        62                          27   
3          Hospital  SLR (NOAA)        22                           0   
2          Hospital   SLR (Our)        22                           9   
5  National Shelter  SLR (NOAA)        47                           0   
4  National Shelter   SLR (Our)        47                          10   
7            School  SLR (NOAA)       171                           0   
6            School   SLR (Our)       171                          59   

   Flood-safe (< 0.10 m)  
1                     62  
0                     35  
3                     22  
2                     13  
5                     47  
4                     37  
7                    171  
6                    112  

Saved: D:\Phd Research\Final_Raster\CI_flood_safe_counts_SLR_vs_NOAA_0p3m.csv


In [5]:
import os
import rasterio
import numpy as np
import geopandas as gpd
import pandas as pd


# -------- USER INPUTS --------
THRESH = 0.3  # meters; depth >= THRESH => flood-affected

rasters = {
    "5-yr":   r"D:\Phd Research\Final_Raster\Bathtub_Depth\bathtub_depth_5yr_mask.tif",
    "20-yr":  r"D:\Phd Research\Final_Raster\Bathtub_Depth\bathtub_depth_20yr_mask.tif",
    "50-yr":  r"D:\Phd Research\Final_Raster\Bathtub_Depth\bathtub_depth_50yr_mask.tif",
    "100-yr": r"D:\Phd Research\Final_Raster\Bathtub_Depth\Bathtub_depth_100yr_surge_SLR.tif",
}

ci_layers = {
    "Fire Station":      r"D:\Phd Research\GIS\Shape\Critical Infrastructure\Fire_stn_study_area.zip",
    "Hospital":          r"D:\Phd Research\GIS\Shape\Critical Infrastructure\hospital_study_area.zip",
    "National Shelter":  r"D:\Phd Research\GIS\Shape\Critical Infrastructure\National_shelter_study_area.zip",
    "School":            r"D:\Phd Research\GIS\Shape\Critical Infrastructure\School_study_area.shp.zip",
}

# Optional: where to save the summary table
out_csv = r"D:\Phd Research\Final_Raster\CI_flood_safe_counts_0.3mthreshold_bathtub.csv"

# -------------------------------------------

def read_ci(path):
    """Read CI points from (zipped) shapefile."""
    p = path
    if p.lower().endswith(".zip") and not p.lower().startswith("zip://"):
        p = f"zip://{p}"
    gdf = gpd.read_file(p)
    # Ensure we have point geometries
    if gdf.empty:
        raise RuntimeError(f"No features found in {path}")
    if not gdf.geometry.geom_type.isin(["Point", "MultiPoint"]).any():
        # Try to explode if multipoints exist
        gdf = gdf.explode(index_parts=False)
        if not gdf.geometry.geom_type.isin(["Point"]).any():
            raise RuntimeError(f"Non-point geometries in {path}. Please provide point CI data.")
    return gdf

def sample_raster_values_at_points(raster_path, gdf_points):
    """Return np.array of sampled raster values at point locations."""
    with rasterio.open(raster_path) as src:
        r_crs = src.crs
        nodata = src.nodata
        # Reproject CI to raster CRS
        pts = gdf_points.to_crs(r_crs).geometry
        coords = [(pt.x, pt.y) for pt in pts]
        # Use the dataset’s sample() method instead of rasterio.sample
        vals = [v[0] for v in src.sample(coords)]
    return np.array(vals, dtype="float64"), nodata


def classify_flooded(values, nodata, thresh):
    """Return a boolean array: True = flood-affected, False = safe."""
    # Treat NoData or non-finite as safe
    is_valid = np.isfinite(values)
    if nodata is not None and np.isfinite(nodata):
        is_valid &= (values != nodata)
    flooded = np.zeros(values.shape, dtype=bool)
    flooded[is_valid] = values[is_valid] >= thresh
    return flooded

# ---- Main analysis ----
records = []

for ci_name, ci_path in ci_layers.items():
    ci_gdf = read_ci(ci_path)
    n_points = len(ci_gdf)

    for scen, rpath in rasters.items():
        vals, nodata = sample_raster_values_at_points(rpath, ci_gdf)
        flooded = classify_flooded(vals, nodata, THRESH)
        n_flood = int(flooded.sum())
        n_safe = int(n_points - n_flood)

        records.append({
            "CI Type": ci_name,
            "Scenario": scen,
            "Total CI": n_points,
            "Flood-affected (>= {:.2f} m)".format(THRESH): n_flood,
            "Flood-safe (< {:.2f} m)".format(THRESH): n_safe
        })

# Make summary table
df = pd.DataFrame.from_records(records)
# Optional: sort
df = df.sort_values(by=["CI Type", "Scenario"])

# Save CSV
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df.to_csv(out_csv, index=False)

# Pretty print
pd.set_option("display.max_rows", 200)
print(df)
print("\nSaved:", out_csv)


             CI Type Scenario  Total CI  Flood-affected (>= 0.30 m)  \
3       Fire Station   100-yr        62                          60   
1       Fire Station    20-yr        62                          46   
0       Fire Station     5-yr        62                          25   
2       Fire Station    50-yr        62                          54   
7           Hospital   100-yr        22                          20   
5           Hospital    20-yr        22                          12   
4           Hospital     5-yr        22                           5   
6           Hospital    50-yr        22                          18   
11  National Shelter   100-yr        47                          46   
9   National Shelter    20-yr        47                          39   
8   National Shelter     5-yr        47                          18   
10  National Shelter    50-yr        47                          42   
15            School   100-yr       171                         169   
13    